In [1]:
import pandas as pd 

In [2]:
data1 = pd.read_csv("../data/cleaned/processing.csv")

In [3]:
def depression_level(score):
    if score <= 9:
        return "Normal"
    elif score <= 13:
        return "Mild"
    elif score <= 20:
        return "Moderate"
    else :
        return "Severe/Extremely Severe"

data1["Depression_Level"] = data1["Depression"].apply(depression_level)

In [4]:
data1.head()

,Age,Gender,Branch,GHQ,Depression,Anxiety,Stress,BRS,Depression_Level
0,19,Male,Computer Science,21,16,12,16,2.33,Moderate
1,17,Female,CSE(Artificial Intelligence),18,14,18,10,1.83,Moderate
2,18,Male,Computer Science,9,0,0,0,0.00,Normal
3,18,Male,Mechanical Engineering,10,0,0,0,0.00,Normal
4,18,Male,Artificial Intelligence and Data Science,3,0,0,0,0.00,Normal


In [5]:
def anxiety_level(score):
    if score <= 7:
        return "Normal"
    elif score <= 9:
        return "Mild"
    elif score <= 14:
        return "Moderate"
    else :
        return "Severe/Extremely Severe"

data1["Anxiety_Level"] = data1["Anxiety"].apply(anxiety_level)

In [6]:
data1.head()

,Age,Gender,Branch,GHQ,Depression,Anxiety,Stress,BRS,Depression_Level,Anxiety_Level
0,19,Male,Computer Science,21,16,12,16,2.33,Moderate,Moderate
1,17,Female,CSE(Artificial Intelligence),18,14,18,10,1.83,Moderate,Severe/Extremely Severe
2,18,Male,Computer Science,9,0,0,0,0.00,Normal,Normal
3,18,Male,Mechanical Engineering,10,0,0,0,0.00,Normal,Normal
4,18,Male,Artificial Intelligence and Data Science,3,0,0,0,0.00,Normal,Normal


In [7]:
def stress_level(score):
    if score <= 14:
        return "Normal"
    elif score <= 18:
        return "Mild"
    elif score <= 25:
        return "Moderate"
    else :
        return "Severe/Extremely Severe"

data1["Stress_Level"] = data1["Stress"].apply(stress_level)

In [8]:
def brs_level(score):
    if score == 0:
        return "Balanced wellbeing"
    elif score < 3.0:
        return "Low"
    elif score <= 4.3:
        return "Average"
    else:
        return "High"

data1["BRS_Level"] = data1["BRS"].apply(brs_level)

In [9]:
data1.head()

,Age,Gender,Branch,GHQ,Depression,Anxiety,Stress,BRS,Depression_Level,Anxiety_Level,Stress_Level,BRS_Level
0,19,Male,Computer Science,21,16,12,16,2.33,Moderate,Moderate,Mild,Low
1,17,Female,CSE(Artificial Intelligence),18,14,18,10,1.83,Moderate,Severe/Extremely Severe,Normal,Low
2,18,Male,Computer Science,9,0,0,0,0.00,Normal,Normal,Normal,Balanced wellbeing
3,18,Male,Mechanical Engineering,10,0,0,0,0.00,Normal,Normal,Normal,Balanced wellbeing
4,18,Male,Artificial Intelligence and Data Science,3,0,0,0,0.00,Normal,Normal,Normal,Balanced wellbeing


In [10]:
def assign_cluster(row):
    
    ghq = row["GHQ"]
    
    dep = row["Depression_Level"]
    anx = row["Anxiety_Level"]
    stress = row["Stress_Level"]
    
    brs = row["BRS_Level"]
    
    # Cluster A
    if ghq < 12:
        return "A"
    
    
    # Count Moderate/Severe Domains
    
    moderate_or_severe = [
        dep in ["Moderate", "Severe/Extremely Severe"],
        anx in ["Moderate", "Severe/Extremely Severe"],
        stress in ["Moderate", "Severe/Extremely Severe"]
    ]
    
    count_mod = sum(moderate_or_severe)
    

    # Check Severe Domains
    
    severe_present = (
        dep in ["Severe/Extremely Severe"] or
        anx in ["Severe/Extremely Severe"] or
        stress in ["Severe/Extremely Severe"]
    )
    

    # Cluster E
    
    if severe_present and brs == "High":
        return "E"
    
    # Cluster F
    
    if severe_present and brs == "Low":
        return "F"

    # Cluster G
    if (
        ghq >= 12 and
        dep in ["Normal", "Mild"] and
        anx in ["Normal", "Mild"] and
        stress in ["Normal", "Mild"] and
        brs == "Low"
    ):
        return "G"


    # Cluster B
    
    if (
        12 <= ghq and
        dep in ["Normal", "Mild"] and
        anx in ["Normal", "Mild"] and
        stress in ["Normal", "Mild"] and
        brs in ["Average", "High"]
    ):
        return "B"
    
    # Cluster H

    if (
        ghq >= 12 and
        count_mod == 1 and
        not severe_present and
        brs == "Low"
    ):
        return "H"
        
    # Cluster C 
    
    if count_mod == 1 and brs in ["Average", "High"]:
        return "C"
    
    # Cluster D    
    
    if count_mod >= 2 and brs in ["Low", "Average"]:
        return "D"
    
    # Default
    return "Unclassified"

In [11]:
data1["Cluster"] = data1.apply(assign_cluster, axis=1)

In [12]:
data1.head()

,Age,Gender,Branch,GHQ,Depression,Anxiety,Stress,BRS,Depression_Level,Anxiety_Level,Stress_Level,BRS_Level,Cluster
0,19,Male,Computer Science,21,16,12,16,2.33,Moderate,Moderate,Mild,Low,D
1,17,Female,CSE(Artificial Intelligence),18,14,18,10,1.83,Moderate,Severe/Extremely Severe,Normal,Low,F
2,18,Male,Computer Science,9,0,0,0,0.00,Normal,Normal,Normal,Balanced wellbeing,A
3,18,Male,Mechanical Engineering,10,0,0,0,0.00,Normal,Normal,Normal,Balanced wellbeing,A
4,18,Male,Artificial Intelligence and Data Science,3,0,0,0,0.00,Normal,Normal,Normal,Balanced wellbeing,A


In [13]:
data1["Cluster"].value_counts()

Cluster
A    2151
D     276
F     256
B     249
C     183
H      73
G      66
E       4
Name: count, dtype: int64

In [14]:
data1.to_csv("../data/cleaned/processing.csv",index=False)